# Sprint 3 — Process Mining
Análise de processos hospitalares com PM4Py sobre o `gold_event_log`.

**Notebook:** `03_process_mining.ipynb`  
**Pipeline de origem:** `gold_transformations`  
**Schema de leitura:** `hospital_santa_rosa.gold_fluxo`  
**Autor:** Ediney Magalhães  
**Criado em:** 2026-06-15

In [0]:
%pip install pm4py
dbutils.library.restartPython()

In [0]:
import pm4py
import pandas as pd

In [0]:
# leitura do event log canônico fo Unity Catalog como Spark DataFrame
df_spark = spark.table("hospital_santa_rosa.gold_fluxo.gold_event_log")

# confirma volume e schema antes de trazer para memória
print(f'Total de registros: {df_spark.count():,}')
df_spark.printSchema()

In [0]:
# converte Spark DataFrame para Pandas
df_pandas = df_spark.toPandas()

# confirma que a conversão manteve o volume correto
print(f'Registros no Pandas: {len(df_pandas):,}')
print(f'Tipo timestamp: {df_pandas['timestamp'].dtype}')

In [0]:
# declarando timestamp em UTC (PM4PY necessita de timestamp com timezone explícito)
df_pandas['timestamp'] = df_pandas['timestamp'].dt.tz_localize('UTC')

# confirmação do tipo correto
print(f'Tipo timestamp após correção: {df_pandas['timestamp'].dtype}')

In [0]:
# preparando DataFrame para PM4PY, declarando quais colunas corresponde ao conceito XES
df_formatado = pm4py.format_dataframe(
    df_pandas,
    case_id='case_id',
    activity_key='activity',
    timestamp_key='timestamp'
)

# confirmando aplicação 
print(f'Colunas após format_dataframe: {list(df_formatado.columns)}')

In [0]:
# verificando registros removidos
total_original = len(df_pandas)
total_formatado = len(df_formatado)
removidos = total_original - total_formatado

print(f"Registros originais:  {total_original:,}")
print(f"Registros formatados: {total_formatado:,}")
print(f"Registros removidos:  {removidos:,}")

# identificando quais colunas tem valores nulos
print("\nNulos por coluna crítica:")
print(f"  case_id:   {df_pandas['case_id'].isna().sum():,}")
print(f"  activity:  {df_pandas['activity'].isna().sum():,}")
print(f"  timestamp: {df_pandas['timestamp'].isna().sum():,}")

In [0]:
# identifica de quais fontes vêm os registros com timestamp nulo
nulos_por_fonte = (
    df_pandas[df_pandas['timestamp'].isna()]
    .groupby('source')
    .size()
    .reset_index(name='registros_nulos')
    .sort_values('registros_nulos', ascending=False)
)

print("Registros com timestamp nulo por fonte:")
print(nulos_por_fonte.to_string(index=False))

In [0]:
# identifica quais atividades específicas têm timestamp nulo por fonte
nulos_por_atividade = (
    df_pandas[df_pandas['timestamp'].isna()]
    .groupby(['source', 'activity'])
    .size()
    .reset_index(name='registros_nulos')
    .sort_values(['source', 'registros_nulos'], ascending=[True, False])
)

print("Atividades com timestamp nulo por fonte:")
print(nulos_por_atividade.to_string(index=False))